In [ ]:
import numpy as np
from tifffile import imread, imwrite
from dataclasses import dataclass

1. Exemplos de NumPy básico

In [ ]:
a = np.array([2, 0, 2, 6])
print(a)
a[:3] = [1, 9, 9]
print(a)
print(a * 2)
print(a - [0, 1, 2, 3])
print(a @ [0, 1, 2, 3])

b = np.zeros((4, 4))
print(b)
# broadcasting
b[:] = a[None]
print(b)
b[:] = a[:, None]
print(b)
print(np.sum(b, axis=0))

2. Processamento Digital de Imagens

In [ ]:
# (linha, coluna, bandas)
pan_img = imread("/images/pan.tif")
rgb_img = imread("/images/rgb.tif")

weights = np.array([2/17, 9/17, 6/17])
sharp_img = rgb_img.repeat(2, axis=0).repeat(2, axis=1)
mask = np.any(sharp_img != 0, axis=-1)
sharp_img[mask] = sharp_img[mask] * (pan_img[mask]/(sharp_img[mask] @ weights))[...,None]
imwrite("/images/sharp.tif", sharp_img)

3. Sensoriamento Remoto

In [ ]:
gsd = 25 # metros
dtm = imread("/images/dem.tif")
area = (slice(1, -1), slice(1, -1))
decl = np.mean(np.abs([
    dtm[*area] - dtm[:-2, area[1]],
    dtm[*area] - dtm[2:,area[1]],
    dtm[*area] - dtm[area[0],2:],
    dtm[*area] - dtm[area[0],:-2]
]), axis=0)/gsd
imwrite("/images/declividade.tif", decl)

4. Ajustamento de Observações

In [ ]:
lb = np.array([1.2, 1.9, 3.2, 3.9, 5.1])
p = np.eye(lb.size)
a = np.column_stack((
    [1, 2, 3, 4, 5],
    np.ones(lb.size)
))
xa = np.linalg.inv(a.T @ p @ a) @ a.T @ p @ lb
la = a @ xa
print("Observações ajustadas:", la)
print("Parâmetros ajustados:", xa)

5. Fotogrametria

In [ ]:
@dataclass
class Model:
  focal: float
  pc: np.ndarray
  opk: np.ndarray

  def rot_mat(self):
    so,sp,sk = np.sin(self.opk)
    co,cp,ck = np.cos(self.opk)
    return np.array([
      [cp*ck,co*sk+so*sp*ck,so*sk-co*sp*ck],
      [-cp*sk,co*ck-so*sp*sk,so*ck+co*sp*sk],
      [sp, -so*cp, co*cp]
    ])

  def project(self, gp: np.ndarray) -> np.ndarray:
    delta = np.transpose(gp - self.pc)
    r = self.rot_mat()
    return np.column_stack([
      r[0] @ delta,
      r[1] @ delta
    ])*-self.focal/(r[2] @ delta)[:,None]

check_points = np.array([
 [583605.701, 7225599.190, 943.977],
 [583499.168, 7225362.328, 936.457],
 [583629.838, 7225202.273, 947.981]
])

model = Model(
  34.145,
  np.array([583807.990, 7225363.334, 1993.541]),
  np.array([
    0.024228120135810105,
    0.009707327029699897,
    -1.2854075799939215
  ])
)
xy = model.project(check_points)
for i in range(check_points.shape[0]):
  print(f"Ponto {i}: {xy[i,0]:.3f}, {xy[i,1]:.3f} mm")